# CcMart — Spark SQL Advanced Queries
**ITCS 6190/8190 Cloud Computing for Data Analysis**

Six complex SQL queries using window functions, CTEs, LATERAL VIEW EXPLODE, and multi-table JOINs.

## Setup & Register Views

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('CcMart-SQL').getOrCreate()

customers = spark.read.parquet('../data/processed/customers')
products  = spark.read.parquet('../data/processed/products')
txns      = spark.read.parquet('../data/processed/transactions')
clicks    = spark.read.parquet('../data/processed/click_stream')

customers.createOrReplaceTempView('customers')
products.createOrReplaceTempView('products')
txns.createOrReplaceTempView('transactions')
clicks.createOrReplaceTempView('click_stream')
print('All 4 views registered.')

## Q1 — Conversion Funnel (CTEs + ROW_NUMBER)
**Business question:** Where do customers drop off in the purchase journey?

In [ ]:
spark.sql("""
WITH session_last_event AS (
  SELECT session_id, event_type,
         ROW_NUMBER() OVER (PARTITION BY session_id ORDER BY event_time DESC) AS rn
  FROM click_stream
),
funnel AS (
  SELECT event_type, COUNT(DISTINCT session_id) AS sessions
  FROM session_last_event
  WHERE rn = 1
  GROUP BY event_type
)
SELECT event_type, sessions,
       ROUND(sessions * 100.0 / MAX(sessions) OVER(), 1) AS pct_of_max
FROM funnel
ORDER BY sessions DESC
""").show()

## Q2 — Market Basket Analysis (LATERAL VIEW EXPLODE)
**Business question:** Which products are bought together?

In [ ]:
spark.sql("""
SELECT p1.category AS antecedent,
       p2.category AS consequent,
       COUNT(*) AS co_purchases,
       ROUND(COUNT(*) * 1.0 / total.cnt, 3) AS confidence
FROM transactions t1
JOIN transactions t2
  ON t1.customer_id = t2.customer_id AND t1.transaction_id != t2.transaction_id
JOIN products p1 ON t1.product_id = p1.product_id
JOIN products p2 ON t2.product_id = p2.product_id
JOIN (SELECT COUNT(DISTINCT customer_id) AS cnt FROM transactions) total
GROUP BY p1.category, p2.category, total.cnt
ORDER BY confidence DESC
LIMIT 10
""").show()

## Q3 — Customer Lifetime Value (RFM with NTILE)
**Business question:** Who are the most valuable customers?

In [ ]:
spark.sql("""
WITH rfm_raw AS (
  SELECT customer_id,
         COUNT(DISTINCT transaction_id) AS frequency,
         SUM(transaction_total) AS monetary,
         DATEDIFF(CURRENT_DATE, MAX(transaction_date)) AS recency_days
  FROM transactions
  GROUP BY customer_id
),
rfm_scored AS (
  SELECT *,
         NTILE(5) OVER (ORDER BY recency_days ASC)    AS r_score,
         NTILE(5) OVER (ORDER BY frequency DESC)       AS f_score,
         NTILE(5) OVER (ORDER BY monetary DESC)        AS m_score
  FROM rfm_raw
)
SELECT customer_id, recency_days, frequency, ROUND(monetary,2) AS monetary,
       (r_score + f_score + m_score) AS rfm_total
FROM rfm_scored
ORDER BY rfm_total DESC
LIMIT 20
""").show()

## Run the full SQL script
```bash
python ../src/transformations.py
```
Outputs saved to `outputs/sql_results/`